# 05 - Anomaly Detection, Regime Classification & Portfolio Risk

This notebook validates and demonstrates the three modules that sit downstream of the core prediction pipeline:

1. **`ResidualAnomalyDetector`** — ensemble anomaly detection on TE prediction residuals.
2. **`RegimeDetector`** — HMM/GMM-based market regime classification with adaptive thresholds.
3. **`PortfolioRiskAggregator`** — portfolio-level TE risk, VaR, sector exposure, and arbitrage aggregation.

**Prerequisites:** run notebook 03 to ensure a trained model artifact exists at `artifacts/te_model.joblib`.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for package in ["yfinance", "plotly", "scipy", "scikit-learn"]:
    spec_name = package.replace("-", "_")
    if importlib.util.find_spec(spec_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from config import (
    DEFAULT_HORIZON,
    DEFAULT_WINDOW,
    ETF_SECTOR_MAP,
    PAIR_CONFIGS,
)
from src.anomaly_detector import ResidualAnomalyDetector
from src.data_loader import MarketDataLoader
from src.features import FeatureEngineer
from src.models import TrackingErrorModel
from src.portfolio_risk import PortfolioRiskAggregator
from src.regime_detector import AdaptiveThresholdConfig, RegimeDetector
from src.utils import time_split

print("Environment ready.")

## 1. Load Data and Compute Residuals

Residuals are defined as:

$$\text{residual}_t = \text{actual\_TE}_t - \hat{\text{TE}}_t$$

All three modules in this notebook operate on this residual series.

In [ ]:
# Load 2-year daily panel
loader = MarketDataLoader()
panel = loader.fetch_universe(PAIR_CONFIGS, period="2y", interval="1d")
features = FeatureEngineer(
    rolling_window=DEFAULT_WINDOW,
    horizon=DEFAULT_HORIZON,
).transform_universe(panel)

# Load trained model
model_path = project_root / "artifacts" / "te_model.joblib"
if not model_path.exists():
    raise FileNotFoundError(
        f"Model artifact not found at {model_path}. Run notebook 03 training save cell first."
    )
model = TrackingErrorModel.load(model_path)

# Score features
features_clean = features.dropna()
feat_cols = model.numeric_columns + model.categorical_columns
features_clean = features_clean.dropna(subset=feat_cols)
x = features_clean[feat_cols]
predictions = model.predict(x)

TARGET_COL = "target_tracking_error"
actual = features_clean[TARGET_COL].values
predicted = predictions
residuals_series = pd.Series(actual - predicted, index=features_clean.index, name="residual")

print(f"Feature rows: {len(features_clean):,}")
print(f"Residual range: [{residuals_series.min():.6f}, {residuals_series.max():.6f}]")
print(f"Residual mean: {residuals_series.mean():.6f}  std: {residuals_series.std():.6f}")

## 2. Residual Anomaly Detection

`ResidualAnomalyDetector` uses three complementary lenses:

| Component | What it catches |
|---|---|
| MLP autoencoder | Unusual joint feature geometry |
| Isolation Forest | Sparse / hard-to-model outliers |
| Statistical shift tests | Regime transitions in mean, volatility, autocorrelation |

In [ ]:
# Use the first pair for illustration
first_pair = features_clean["pair"].iloc[0]
pair_mask = features_clean["pair"] == first_pair
pair_actual = pd.Series(actual, index=features_clean.index)[pair_mask]
pair_predicted = pd.Series(predicted, index=features_clean.index)[pair_mask]

print(f"Running anomaly detection on pair: {first_pair} ({len(pair_actual)} rows)")

detector = ResidualAnomalyDetector(
    short_window=12,
    long_window=36,
    contamination=0.05,
    random_state=42,
)
scored = detector.fit_score(
    actual_tracking_error=pair_actual,
    predicted_tracking_error=pair_predicted,
)

n_anomalies = int(scored["anomaly_detected"].sum())
pct_anomalies = 100.0 * n_anomalies / len(scored)
print(f"\nAnomaly summary for {first_pair}:")
print(f"  Total scored rows : {len(scored)}")
print(f"  Anomalies detected: {n_anomalies} ({pct_anomalies:.1f}%)")
print(f"\nAnomaly type distribution:")
print(scored["anomaly_type"].value_counts().to_string())

In [ ]:
# Ensemble score over time with anomaly flags
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=scored.index,
    y=scored["ensemble_score"],
    mode="lines",
    name="Ensemble Score",
    line=dict(color="steelblue", width=1),
))

anomaly_rows = scored[scored["anomaly_detected"] == True]
fig.add_trace(go.Scatter(
    x=anomaly_rows.index,
    y=anomaly_rows["ensemble_score"],
    mode="markers",
    name="Anomaly",
    marker=dict(color="crimson", size=7, symbol="x"),
))

fig.update_layout(
    title=f"Residual Anomaly Ensemble Score — {first_pair}",
    xaxis_title="Date",
    yaxis_title="Ensemble Score",
    legend=dict(orientation="h"),
    height=400,
)
fig.show()

In [ ]:
# Confidence distribution by anomaly type
fig2 = px.box(
    scored,
    x="anomaly_type",
    y="confidence",
    color="anomaly_type",
    title=f"Anomaly Confidence Distribution by Type — {first_pair}",
    labels={"anomaly_type": "Anomaly Type", "confidence": "Confidence"},
    height=400,
)
fig2.show()

# Latest anomaly result
latest = detector.latest_result(
    actual_tracking_error=pair_actual,
    predicted_tracking_error=pair_predicted,
)
print("\nLatest anomaly result:")
for k, v in latest.items():
    print(f"  {k}: {v}")

## 3. Market Regime Detection

`RegimeDetector` fits a Hidden Markov Model (with GMM fallback) on rolling residual features to classify each observation into one of three regimes:

| Regime | Characteristics |
|---|---|
| **Calm** | Low residual volatility, stable mean, weak autocorrelation |
| **Stress** | Elevated volatility, directional mean drift |
| **High_Vol** | Extreme residual swings, high kurtosis, strong persistence |

Each regime produces adapted operational thresholds (alert TE level, minimum arbitrage confidence, anomaly sensitivity).

In [ ]:
pair_residuals = pair_actual.values - pair_predicted

regime_detector = RegimeDetector(
    rolling_window=24,
    smoothing_alpha=0.35,
    min_regime_persistence=4,
    base_thresholds=AdaptiveThresholdConfig(
        alert_tracking_error=0.80,
        arbitrage_confidence_min=0.70,
        anomaly_reconstruction_threshold=0.050,
    ),
    random_state=42,
)

result = regime_detector.detect_regime(pair_residuals, history_points=100)

print(f"Current regime     : {result['current_regime']}")
print(f"Confidence         : {result['confidence']:.4f}")
print(f"Model used         : {result['model_used']}")
print(f"\nRegime probabilities:")
for regime, prob in result["regime_probability"].items():
    print(f"  {regime:<10}: {prob:.4f}")
print(f"\nAdaptive thresholds:")
for k, v in result["adaptive_thresholds"].items():
    print(f"  {k}: {v}")
print(f"\nExplanation: {result['explanation']}")

In [ ]:
# Regime history visualization
history = pd.DataFrame(result["regime_history"])
history["timestamp"] = pd.to_datetime(history["timestamp"])

regime_color_map = {"Calm": "#2ecc71", "Stress": "#e67e22", "High_Vol": "#e74c3c"}

fig3 = px.scatter(
    history,
    x="timestamp",
    y="confidence",
    color="regime",
    color_discrete_map=regime_color_map,
    title=f"Regime History (last {len(history)} observations) — {first_pair}",
    labels={"timestamp": "Date", "confidence": "Regime Confidence", "regime": "Regime"},
    height=400,
)
fig3.update_traces(marker=dict(size=6))
fig3.show()

print(f"\nRegime distribution in history:")
print(history["regime"].value_counts().to_string())

In [ ]:
# Run regime detection across all pairs and compare current regimes
regime_summary_rows = []

for pair_id in features_clean["pair"].unique():
    mask = features_clean["pair"] == pair_id
    p_actual = pd.Series(actual, index=features_clean.index)[mask]
    p_pred = pd.Series(predicted, index=features_clean.index)[mask]
    p_resid = p_actual.values - p_pred

    if len(p_resid) < 60:
        continue

    try:
        r = regime_detector.detect_regime(p_resid, history_points=30)
        regime_summary_rows.append({
            "pair": pair_id,
            "current_regime": r["current_regime"],
            "confidence": round(r["confidence"], 4),
            "model_used": r["model_used"],
            "alert_te_threshold": r["adaptive_thresholds"]["alert_tracking_error"],
            "min_arb_confidence": r["adaptive_thresholds"]["arbitrage_confidence_min"],
        })
    except Exception as exc:
        print(f"  Skipped {pair_id}: {exc}")

regime_summary = pd.DataFrame(regime_summary_rows)
print("Regime summary across pairs:")
print(regime_summary.to_string(index=False))

## 4. Portfolio-Level Risk Aggregation

`PortfolioRiskAggregator` combines per-pair predictions and signals into portfolio-wide metrics:

| Metric | Description |
|---|---|
| **Total predicted TE (%)** | Weighted average predicted tracking error across all ETF positions |
| **Portfolio TE VaR (%)** | Historical VaR of portfolio-level TE at a given confidence level |
| **Arbitrage risk score** | Weighted confidence × TE intensity across active signals |
| **Actionable weight (%)** | Portfolio weight with active CREATE or REDEEM signals |

In [ ]:
from src.arbitrage_signal import ArbitrageSignalGenerator

# Build a live_overview-style DataFrame (latest row per pair)
live_rows = []
for pair_id in features_clean["pair"].unique():
    mask = features_clean["pair"] == pair_id
    pair_feats = features_clean[mask]
    pair_preds = pd.Series(predicted, index=features_clean.index)[mask]
    if pair_feats.empty:
        continue

    latest_feat = pair_feats.iloc[-1].to_dict()
    latest_pred = float(pair_preds.iloc[-1])
    etf_ticker = pair_id.split("_")[0]

    live_rows.append({
        "pair": pair_id,
        "etf_ticker": etf_ticker,
        "predicted_tracking_error": latest_pred,
        "uncertainty_sigma": abs(latest_pred) * 0.15,  # heuristic 15% sigma proxy
        "risk_bucket": "High" if abs(latest_pred) > 0.001 else "Low",
    })

live_overview = pd.DataFrame(live_rows)
print("Live overview (latest per pair):")
print(live_overview[["pair", "etf_ticker", "predicted_tracking_error", "risk_bucket"]].to_string(index=False))

In [ ]:
# Build a prediction time series for VaR computation
pred_series_rows = []
for pair_id in features_clean["pair"].unique():
    mask = features_clean["pair"] == pair_id
    pair_preds = pd.Series(predicted, index=features_clean.index)[mask]
    etf_ticker = pair_id.split("_")[0]
    for ts, val in pair_preds.items():
        pred_series_rows.append({
            "pair": pair_id,
            "etf_ticker": etf_ticker,
            "timestamp": ts,
            "predicted_tracking_error": float(val),
            "uncertainty_sigma": abs(float(val)) * 0.15,
        })

prediction_series = pd.DataFrame(pred_series_rows)
print(f"Prediction series shape: {prediction_series.shape}")

In [ ]:
# Generate universe signals using ArbitrageSignalGenerator
signal_gen = ArbitrageSignalGenerator()
signal_rows = []
for pair_id in features_clean["pair"].unique():
    mask = features_clean["pair"] == pair_id
    pair_feats = features_clean[mask]
    pair_preds_vals = pd.Series(predicted, index=features_clean.index)[mask]
    if len(pair_feats) < 5:
        continue

    try:
        signal = signal_gen.generate(
            pair=pair_id,
            features=pair_feats,
            predictions=pair_preds_vals,
        )
        signal["pair"] = pair_id
        signal["etf_ticker"] = pair_id.split("_")[0]
        signal["predicted_tracking_error"] = float(pair_preds_vals.iloc[-1])
        signal_rows.append(signal)
    except Exception as exc:
        print(f"  Skipped signal for {pair_id}: {exc}")

universe_signals = pd.DataFrame(signal_rows)
if not universe_signals.empty:
    print("Universe signals:")
    cols = [c for c in ["pair", "action", "confidence", "recommended_shares", "estimated_net_profit_usd"] if c in universe_signals.columns]
    print(universe_signals[cols].to_string(index=False))

In [ ]:
# Define equal-weight portfolio for illustration
etf_tickers = [pair_id.split("_")[0] for pair_id in features_clean["pair"].unique()]
raw_weights = {etf: 1.0 for etf in set(etf_tickers)}

aggregator = PortfolioRiskAggregator(var_confidence=0.95, var_lookback=240)
normalized_weights = aggregator.normalize_weights(raw_weights)

print("Normalized portfolio weights (equal-weight):")
for etf, w in normalized_weights.items():
    print(f"  {etf:<10}: {w:.4f}")

In [ ]:
# Portfolio TE time series and VaR
latest_predictions_df = live_overview.copy()
latest_predictions_df["timestamp"] = features_clean.index[-1]

portfolio_series = aggregator.build_portfolio_prediction_series(
    prediction_series=prediction_series,
    latest_predictions=latest_predictions_df,
    normalized_weights=normalized_weights,
)

var_pct = aggregator.compute_var_pct(portfolio_series)
print(f"Portfolio 95% TE VaR: {var_pct:.4f}%")
print(f"Portfolio series rows: {len(portfolio_series)}")

if not portfolio_series.empty:
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(
        x=portfolio_series["timestamp"],
        y=portfolio_series["portfolio_predicted_te"] * 100,
        mode="lines",
        name="Portfolio TE (%)",
        line=dict(color="royalblue"),
    ))
    if not np.isnan(var_pct):
        fig4.add_hline(
            y=var_pct,
            line_dash="dash",
            line_color="crimson",
            annotation_text=f"95% VaR: {var_pct:.4f}%",
        )
    fig4.update_layout(
        title="Portfolio Predicted Tracking Error Over Time",
        xaxis_title="Date",
        yaxis_title="Portfolio TE (%)",
        height=400,
    )
    fig4.show()

In [ ]:
# ETF risk contributions
contributions = aggregator.compute_etf_contributions(
    live_overview=live_overview,
    normalized_weights=normalized_weights,
)

if not contributions.empty:
    print("ETF risk contributions:")
    print(contributions[["etf_ticker", "weight", "weighted_abs_te", "risk_contribution_pct"]].to_string(index=False))

    fig5 = px.bar(
        contributions,
        x="etf_ticker",
        y="risk_contribution_pct",
        color="risk_bucket",
        title="ETF Risk Contribution to Portfolio TE (%)",
        labels={"etf_ticker": "ETF", "risk_contribution_pct": "Risk Contribution (%)"},
        height=400,
    )
    fig5.show()

In [ ]:
# Sector exposure
sector_exposure = aggregator.compute_sector_exposure(
    normalized_weights=normalized_weights,
    sector_map=ETF_SECTOR_MAP,
)

print("Sector exposure:")
print(sector_exposure.to_string(index=False))

fig6 = px.pie(
    sector_exposure,
    names="sector",
    values="exposure_pct",
    title="Portfolio Sector Exposure (%)",
    height=400,
)
fig6.show()

In [ ]:
# Full portfolio risk summary
if not universe_signals.empty:
    summary = aggregator.summarize(
        live_overview=live_overview,
        portfolio_series=portfolio_series,
        universe_signals=universe_signals,
        normalized_weights=normalized_weights,
    )

    print("Portfolio Risk Summary")
    print("=" * 40)
    print(f"  Total predicted TE      : {summary.total_predicted_te_pct:.4f}%")
    print(f"  Portfolio TE VaR (95%)  : {summary.portfolio_te_var_pct:.4f}%")
    print(f"  Arb risk score          : {summary.aggregate_arbitrage_risk_score:.4f}")
    print(f"  Actionable weight       : {summary.actionable_weight_pct:.2f}%")
else:
    print("No universe signals available — skipping full summary.")

## 5. Summary

This notebook demonstrated the full post-prediction analytics layer:

| Module | Output |
|---|---|
| `ResidualAnomalyDetector` | Per-timestamp anomaly flags, types, confidence scores, and explanations |
| `RegimeDetector` | Current market regime (Calm/Stress/High_Vol) with adaptive thresholds per pair |
| `PortfolioRiskAggregator` | Portfolio TE time series, 95% VaR, ETF contributions, sector exposure, arb risk score |

These outputs feed directly into the Streamlit dashboard (`app.py`) for live monitoring.